# D. Community Questions

## Aim

The aim of this notebook is to turn the community questions PDF into a clear table that we can use later to find relevant web pages.

We keep each question and support item with its section heading and page number. We then add the heading to the text used for searching, so short questions are easier to understand.


## 1. Read the PDF

Read each page with `pypdf`, keeping the headings and bullet points in place. Display the text below to see what was extracted.


In [1]:
from pathlib import Path
import re

import pandas as pd
from pypdf import PdfReader

PDF_PATH = Path("../docs/CSN Community Questions Dataset.pdf")
reader = PdfReader(PDF_PATH)
pages_df = pd.DataFrame([
    {
        "page_number": page_number,
        "page_text": page.extract_text(extraction_mode="layout") or "",
    }
    for page_number, page in enumerate(reader.pages, start=1)
])

if pages_df.empty or pages_df["page_text"].str.strip().eq("").any():
    raise ValueError("One or more PDF pages have no extractable text; review the PDF.")

print(f"Source: {PDF_PATH.name}")
print(f"Pages extracted: {len(pages_df)}")
for page in pages_df.itertuples(index=False):
    print(f"\n--- Page {page.page_number} ---\n{page.page_text}")

Source: CSN Community Questions Dataset.pdf
Pages extracted: 2

--- Page 1 ---
Understanding Your Care Pathway in the Netherlands

The goal is to help patients actively navigate their care journey once they are in the
system.

Suggested guiding questions

1. Referrals and process

    ●   How do referrals work in the Netherlands?
    ●   Can I choose my hospital or specialist?
    ●   Can I change doctors?

2. Waiting times

    ●   How long should I expect to wait for appointments or treatment?
    ●   What can I do if I feel the process is too slow?
    ●   Is it normal to wait between steps?

3. Treatment decisions

    ●   How are treatment decisions made?
    ●   Can I ask for alternative options?
    ●   Do I have time to decide?

4. Second opinions

    ●   Am I allowed to get a second opinion? Also in another country.
    ●   How do I request one?
    ●   Will it be covered by insurance?

5. Additional care and support

    ●   What additional services can I access?
    ●   Psy

## 2. Put the questions and support items into a table

Each bullet point becomes one row. Join wrapped lines so the full text stays together, and keep each item under its original heading, even when it continues onto the next page.

Items with a question mark are labelled `question`. The other bullets, such as “Physiotherapy”, are labelled `support_item`.


In [2]:
HEADING_PATTERN = re.compile(r"^(\d+)\.\s+(.+)$")
BULLET_PATTERN = re.compile(r"^[●•▪]\s*(.*)$")


def clean_text(text):
    return re.sub(r"\s+", " ", text).strip()


def extract_items(pages):
    section_rows = []
    item_rows = []
    current_section = None
    pending_item = None

    def finish_item():
        nonlocal pending_item
        if pending_item is None:
            return
        text = clean_text(" ".join(pending_item.pop("text_parts")))
        if not text:
            raise ValueError("Found an empty bullet; review the extracted page text.")
        item_rows.append({
            "item_id": len(item_rows) + 1,
            **pending_item,
            "item_type": "question" if "?" in text else "support_item",
            "item_text": text,
        })
        pending_item = None

    for page in pages.sort_values("page_number").itertuples(index=False):
        for raw_line in page.page_text.splitlines():
            line = clean_text(raw_line)
            if not line:
                continue

            heading = HEADING_PATTERN.match(line)
            if heading:
                finish_item()
                current_section = {
                    "section_number": int(heading.group(1)),
                    "section_heading": heading.group(2),
                    "section_page_number": page.page_number,
                }
                section_rows.append(current_section.copy())
                continue

            bullet = BULLET_PATTERN.match(line)
            if bullet:
                finish_item()
                if current_section is None:
                    raise ValueError(f"Bullet before any section on page {page.page_number}.")
                pending_item = {
                    **current_section,
                    "page_number": page.page_number,
                    "page_end_number": page.page_number,
                    "text_parts": [bullet.group(1)],
                }
                continue

            if pending_item is not None:
                pending_item["text_parts"].append(line)
                pending_item["page_end_number"] = page.page_number
            elif current_section is not None:
                raise ValueError(
                    f"Unexpected text after a heading on page {page.page_number}: {line}"
                )
            # The introduction stays in pages_df; only bullet items go into the table.

    finish_item()
    if not section_rows or not item_rows:
        raise ValueError("No sections or bullet items found; review the PDF layout.")
    return pd.DataFrame(section_rows), pd.DataFrame(item_rows)


sections_df, items_df = extract_items(pages_df)
print(f"Sections: {len(sections_df)}")
print(f"Items extracted: {len(items_df)}")
sections_df


Sections: 6
Items extracted: 21


,section_number,section_heading,section_page_number
0,1,Referrals and process,1
1,2,Waiting times,1
2,3,Treatment decisions,1
3,4,Second opinions,1
4,5,Additional care and support,1
5,6,Communication and understanding,1


## 3. Add context for searching

Some questions need their heading to make sense. For example, “How do I request one?” refers to a second opinion.

Create `search_text` by joining the heading and item text: “Second opinions. How do I request one?” Keep the original wording in `item_text`.


In [3]:
items_df["search_text"] = items_df["section_heading"] + ". " + items_df["item_text"]

questions_df = items_df.loc[items_df["item_type"].eq("question")].copy()
support_items_df = items_df.loc[items_df["item_type"].eq("support_item")].copy()

print(f"Questions: {len(questions_df)}")
print(f"Support items: {len(support_items_df)}")
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(items_df[["item_id", "item_text", "search_text"]])


Questions: 17
Support items: 4


,item_id,item_text,search_text
0,1,How do referrals work in the Netherlands?,Referrals and process. How do referrals work in the Netherlands?
1,2,Can I choose my hospital or specialist?,Referrals and process. Can I choose my hospital or specialist?
2,3,Can I change doctors?,Referrals and process. Can I change doctors?
3,4,How long should I expect to wait for appointments or treatment?,Waiting times. How long should I expect to wait for appointments or treatment?
4,5,What can I do if I feel the process is too slow?,Waiting times. What can I do if I feel the process is too slow?
5,6,Is it normal to wait between steps?,Waiting times. Is it normal to wait between steps?
6,7,How are treatment decisions made?,Treatment decisions. How are treatment decisions made?
7,8,Can I ask for alternative options?,Treatment decisions. Can I ask for alternative options?
8,9,Do I have time to decide?,Treatment decisions. Do I have time to decide?
9,10,Am I allowed to get a second opinion? Also in another country.,Second opinions. Am I allowed to get a second opinion? Also in another country.


## 4. Review the results

Review the full table and the number of questions and support items in each section.

The page columns show where each item and its heading came from:

- `page_number`: the page where the item starts.
- `page_end_number`: the page where the item ends.
- `section_page_number`: the page where its heading appears.

For example, “Communication and understanding” starts on page 1, while its questions are on page 2.


In [4]:
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(items_df)

,item_id,section_number,section_heading,section_page_number,page_number,page_end_number,item_type,item_text,search_text
0,1,1,Referrals and process,1,1,1,question,How do referrals work in the Netherlands?,Referrals and process. How do referrals work in the Netherlands?
1,2,1,Referrals and process,1,1,1,question,Can I choose my hospital or specialist?,Referrals and process. Can I choose my hospital or specialist?
2,3,1,Referrals and process,1,1,1,question,Can I change doctors?,Referrals and process. Can I change doctors?
3,4,2,Waiting times,1,1,1,question,How long should I expect to wait for appointments or treatment?,Waiting times. How long should I expect to wait for appointments or treatment?
4,5,2,Waiting times,1,1,1,question,What can I do if I feel the process is too slow?,Waiting times. What can I do if I feel the process is too slow?
5,6,2,Waiting times,1,1,1,question,Is it normal to wait between steps?,Waiting times. Is it normal to wait between steps?
6,7,3,Treatment decisions,1,1,1,question,How are treatment decisions made?,Treatment decisions. How are treatment decisions made?
7,8,3,Treatment decisions,1,1,1,question,Can I ask for alternative options?,Treatment decisions. Can I ask for alternative options?
8,9,3,Treatment decisions,1,1,1,question,Do I have time to decide?,Treatment decisions. Do I have time to decide?
9,10,4,Second opinions,1,1,1,question,Am I allowed to get a second opinion? Also in another country.,Second opinions. Am I allowed to get a second opinion? Also in another country.


In [5]:
section_summary_df = (
    items_df.groupby(["section_number", "section_heading", "item_type"])
    .size()
    .unstack("item_type", fill_value=0)
    .reindex(columns=["question", "support_item"], fill_value=0)
    .rename(columns={"question": "question_count", "support_item": "support_item_count"})
    .reset_index()
)
section_summary_df

item_type,section_number,section_heading,question_count,support_item_count
0,1,Referrals and process,3,0
1,2,Waiting times,3,0
2,3,Treatment decisions,3,0
3,4,Second opinions,3,0
4,5,Additional care and support,2,4
5,6,Communication and understanding,3,0


## 5. Save the search items

Save the full table as `community_search_items.csv` so the next notebook can load it for search and filtering. The `item_type` column lets us select questions or support items when needed.


In [31]:
OUTPUT_PATH = Path("../data/interim/community_search_items.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

items_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(items_df)} search items to {OUTPUT_PATH}")


Saved 21 search items to ../data/interim/community_search_items.csv


## What we have so far

`community_search_items.csv` contains all questions and support items, with their headings, page numbers and search text.

The next step is to prepare the candidate URL text for matching against these items.
